# 05 — GroupBy & Aggregations

## The Restaurant Analytics Analogy

Imagine you run a chain of restaurants and you have a giant spreadsheet of every order ever placed. You want to answer:
- *How much revenue did each branch make?*
- *What's the average order value per category?*
- *Which branch-category combo sells the most?*

You'd group rows by branch, then apply math (sum, average, count) to each group. That's **groupBy + agg**.

In Spark, `groupBy()` triggers a **shuffle** (wide transformation) — data with the same key moves to the same partition. Then `agg()` applies aggregation functions on each group **in parallel**.

---

## What You'll Learn
1. `groupBy()` + `agg()` pattern
2. All built-in aggregation functions
3. Multiple aggregations in one pass
4. `pivot()` — turn row values into columns
5. `rollup()` and `cube()` for subtotals
6. Window functions preview (full coverage in Week 5)
7. Performance notes: `groupBy` shuffle cost

In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.functions import (
    col, count, countDistinct, approx_count_distinct,
    sum, avg, mean, min, max,
    collect_list, collect_set,
    stddev, variance, first, last,
    round, lit, when
)

spark = SparkSession.builder \
    .appName("GroupBy_Aggregations") \
    .master("local[*]") \
    .config("spark.sql.shuffle.partitions", "4") \
    .getOrCreate()

spark.sparkContext.setLogLevel("ERROR")
print(f"Spark {spark.version} ready")

## 1. Sample Data

We'll use a richer sales dataset this week with multiple dimensions to aggregate across.

In [ ]:
from pyspark.sql.types import StructType, StructField, IntegerType, StringType, DoubleType, DateType

sales_data = [
    (1,  "Alice",   "Engineering", 95000.0, "US", 2021),
    (2,  "Bob",     "Engineering", 88000.0, "US", 2020),
    (3,  "Carol",   "Marketing",   72000.0, "UK", 2019),
    (4,  "David",   "Marketing",   68000.0, "UK", 2022),
    (5,  "Eve",     "HR",          58000.0, "IN", 2021),
    (6,  "Frank",   "Engineering", 102000.0, "US", 2018),
    (7,  "Grace",   "HR",          61000.0, "IN", 2020),
    (8,  "Heidi",   "Marketing",   75000.0, "US", 2019),
    (9,  "Ivan",    "Engineering", 91000.0, "IN", 2022),
    (10, "Judy",    "HR",          55000.0, "UK", 2023),
    (11, "Karl",    "Engineering", 97000.0, "UK", 2021),
    (12, "Laura",   "Marketing",   69000.0, "IN", 2020),
    (13, "Mallory", "HR",          None,    "US", 2022),   # null salary
    (14, "Niaj",    "Engineering", 85000.0, "IN", 2021),
    (15, "Olivia",  "Marketing",   78000.0, "US", 2023),
]

schema = StructType([
    StructField("id",         IntegerType(), False),
    StructField("name",       StringType(),  False),
    StructField("dept",       StringType(),  False),
    StructField("salary",     DoubleType(),  True),   # nullable — Mallory has None
    StructField("country",    StringType(),  False),
    StructField("hire_year",  IntegerType(), False),
])

df = spark.createDataFrame(sales_data, schema=schema)
df.printSchema()
df.show()

## 2. Basic groupBy + agg

```
df.groupBy("col1", "col2")   ← GroupedData object (lazy, no shuffle yet)
  .agg(func1, func2, ...)     ← triggers shuffle → action on result
```

**Key insight:** `groupBy()` returns a `GroupedData` object — it's **lazy**. The shuffle only happens when you call `.agg()`, `.count()`, `.sum()`, etc.

In [ ]:
# --- Most common pattern: groupBy + agg with named columns ---

dept_stats = df.groupBy("dept").agg(
    count("*").alias("headcount"),
    round(avg("salary"), 2).alias("avg_salary"),
    min("salary").alias("min_salary"),
    max("salary").alias("max_salary"),
    sum("salary").alias("total_salary"),
)

dept_stats.orderBy("dept").show()

In [ ]:
# --- Group by multiple columns ---

dept_country = df.groupBy("dept", "country").agg(
    count("*").alias("n"),
    round(avg("salary"), 0).alias("avg_salary"),
).orderBy("dept", "country")

dept_country.show()

## 3. Shorthand Aggregation Methods

`GroupedData` has shorthand methods for the most common ops. These are less flexible (no alias, one function at a time) but faster to type.

In [ ]:
# .count() — counts rows per group (not null-aware, counts ALL rows)
df.groupBy("dept").count().show()

# .sum("col") — sum of one column
df.groupBy("dept").sum("salary").show()

# .avg("col")
df.groupBy("dept").avg("salary").show()

# .min() / .max()
df.groupBy("dept").min("salary").show()

# LIMITATION: cannot alias, cannot combine multiple aggs in one call
# Always prefer .agg() for real work

## 4. All Aggregation Functions

| Function | Description | Null handling |
|---|---|---|
| `count("*")` | Count all rows (including nulls) | includes nulls |
| `count("col")` | Count non-null values in column | skips nulls |
| `countDistinct("col")` | Count distinct non-null values | exact, expensive |
| `approx_count_distinct("col", rsd)` | Approximate distinct count (HyperLogLog) | ~2% error, fast |
| `sum("col")` | Sum of values | skips nulls |
| `avg("col")` / `mean()` | Mean of values | skips nulls |
| `min("col")` | Minimum value | skips nulls |
| `max("col")` | Maximum value | skips nulls |
| `stddev("col")` | Sample standard deviation | skips nulls |
| `variance("col")` | Sample variance | skips nulls |
| `first("col", ignorenulls)` | First value in group | by partition order |
| `last("col", ignorenulls)` | Last value in group | by partition order |
| `collect_list("col")` | List of all values (with dups) | skips nulls |
| `collect_set("col")` | Set of unique values | skips nulls |

In [ ]:
# --- count("*") vs count("col") — critical difference ---

df.groupBy("dept").agg(
    count("*").alias("total_rows"),          # counts ALL rows (including Mallory with null salary)
    count("salary").alias("non_null_salary"), # skips null → Mallory excluded from HR count
).orderBy("dept").show()

# Notice: HR has 3 total_rows but only 2 non_null_salary (Mallory has null)

In [ ]:
# --- countDistinct vs approx_count_distinct ---
# On small data, use countDistinct. On 100M+ rows, use approx_count_distinct.

df.groupBy("dept").agg(
    countDistinct("country").alias("distinct_countries_exact"),
    approx_count_distinct("country", rsd=0.05).alias("distinct_countries_approx"),
).orderBy("dept").show()

# rsd = relative standard deviation (0.05 = ~5% error, uses less memory)

In [ ]:
# --- stddev, variance ---

df.groupBy("dept").agg(
    round(avg("salary"), 2).alias("avg_salary"),
    round(stddev("salary"), 2).alias("salary_stddev"),
    round(variance("salary"), 2).alias("salary_variance"),
).orderBy("dept").show()

In [ ]:
# --- collect_list vs collect_set ---
# collect_list: ALL values including duplicates → ArrayType
# collect_set:  UNIQUE values only → ArrayType (unordered!)

df.groupBy("dept").agg(
    collect_set("country").alias("countries_present"),   # unique countries per dept
    collect_list("name").alias("employee_names"),         # all names in that dept
).orderBy("dept").show(truncate=False)

# WARNING: collect_list on huge groups → OOM. Use only when groups are small.

In [ ]:
# --- first / last ---
# NOT deterministic unless you orderBy before groupBy!
# "first" within a group depends on partition order.

df.groupBy("dept").agg(
    first("name", ignorenulls=True).alias("first_employee"),
    last("name", ignorenulls=True).alias("last_employee"),
    first("salary", ignorenulls=True).alias("first_salary"),
).orderBy("dept").show()

# For deterministic "first": use Window functions with orderBy (covered in Week 5)

## 5. Chained / Conditional Aggregations

You can combine `when()` inside aggregation functions to do **conditional aggregations** — the Spark equivalent of SQL's `SUM(CASE WHEN ...)` pattern.

In [ ]:
# --- Conditional aggregation: count employees above avg salary ---
# SQL equivalent: COUNT(CASE WHEN salary > 80000 THEN 1 END)

df.groupBy("dept").agg(
    count("*").alias("total"),
    count(when(col("salary") > 80000, 1)).alias("high_earners"),
    count(when(col("salary") <= 80000, 1)).alias("mid_earners"),
    round(
        count(when(col("salary") > 80000, 1)) / count("*") * 100, 1
    ).alias("pct_high_earners"),
).orderBy("dept").show()

In [ ]:
# --- sum of salary by hire year cohort ---

df.groupBy("dept").agg(
    sum(when(col("hire_year") <= 2020, col("salary"))).alias("salary_pre2021"),
    sum(when(col("hire_year") >= 2021, col("salary"))).alias("salary_2021plus"),
).orderBy("dept").show()

## 6. pivot() — Rows to Columns

**Pivot** takes unique values from one column and turns them into separate column headers.

```
Before pivot (long format):          After pivot (wide format):
dept       country  avg_salary       dept         IN       UK       US
Eng        IN       91000            Eng          91000    97000    94666
Eng        UK       97000            HR           61000    55000    null
Eng        US       94666            Marketing    69000    72000    74500
HR         IN       61000
...
```

Think of it like an **Excel PivotTable** that reshapes data.

In [ ]:
# --- Basic pivot: avg salary per dept per country ---

pivot_df = df.groupBy("dept") \
    .pivot("country") \
    .agg(round(avg("salary"), 0))

pivot_df.show()
pivot_df.printSchema()

In [ ]:
# --- Pivot with explicit values (ALWAYS do this in production!) ---
# Without explicit values, Spark runs an extra job to discover unique values.
# Providing them upfront saves one full scan of the data.

pivot_explicit = df.groupBy("dept") \
    .pivot("country", ["IN", "UK", "US"]) \
    .agg(round(avg("salary"), 0))

pivot_explicit.show()

# Best practice: always supply the values list when you know them

In [ ]:
# --- Pivot with multiple aggregations per pivoted column ---
# Each agg gets a column: country_agg (e.g. IN_avg, IN_count)

multi_agg_pivot = df.groupBy("dept") \
    .pivot("country", ["IN", "UK", "US"]) \
    .agg(
        round(avg("salary"), 0).alias("avg"),
        count("*").alias("n"),
    )

multi_agg_pivot.show(truncate=False)

## 7. rollup() — Hierarchical Subtotals

`rollup()` computes subtotals at every level of a column hierarchy, **plus a grand total**.

```
rollup("dept", "country") computes:
  (dept, country)  ← leaf level
  (dept, null)     ← subtotal per dept
  (null, null)     ← grand total
```

Nulls in the grouping columns indicate the subtotal/grand total rows.

In [ ]:
from pyspark.sql.functions import coalesce

rollup_df = df.rollup("dept", "country").agg(
    count("*").alias("headcount"),
    round(avg("salary"), 0).alias("avg_salary"),
).orderBy("dept", "country")

rollup_df.show()

# Rows where dept=null, country=null → grand total row
# Rows where country=null but dept is set → dept subtotal

In [ ]:
# --- Make rollup output readable using coalesce for labels ---

rollup_labeled = df.rollup("dept", "country").agg(
    count("*").alias("headcount"),
    round(avg("salary"), 0).alias("avg_salary"),
).withColumn(
    "dept",    coalesce(col("dept"),    lit("ALL DEPTS"))
).withColumn(
    "country", coalesce(col("country"), lit("ALL COUNTRIES"))
).orderBy("dept", "country")

rollup_labeled.show()

## 8. cube() — All Dimension Combinations

`cube()` is like `rollup()` but computes subtotals for **every possible combination** of the grouping columns.

```
cube("dept", "country") computes:
  (dept, country)  ← all specific combos
  (dept, null)     ← subtotals per dept
  (null, country)  ← subtotals per country  ← rollup doesn't have this!
  (null, null)     ← grand total
```

| | rollup | cube |
|---|---|---|
| Leaf combos | ✅ | ✅ |
| Left subtotals (dept only) | ✅ | ✅ |
| Right subtotals (country only) | ❌ | ✅ |
| Grand total | ✅ | ✅ |

In [ ]:
cube_df = df.cube("dept", "country").agg(
    count("*").alias("headcount"),
    round(avg("salary"), 0).alias("avg_salary"),
).orderBy("dept", "country")

cube_df.show(30)

# vs rollup — cube has extra rows where dept=null but country is set

In [ ]:
# --- Filter cube result to just country totals ---

country_totals = cube_df \
    .where(col("dept").isNull() & col("country").isNotNull()) \
    .orderBy("country")

country_totals.show()
# This is only possible with cube, not rollup

## 9. Using SQL for Aggregations

SQL is often more readable for complex GROUP BY queries. The Catalyst optimizer produces identical plans for both.

In [ ]:
df.createOrReplaceTempView("employees")

# GROUP BY with HAVING (filter groups after aggregation)
spark.sql("""
    SELECT
        dept,
        COUNT(*) AS headcount,
        ROUND(AVG(salary), 2) AS avg_salary,
        SUM(salary) AS total_salary
    FROM employees
    WHERE salary IS NOT NULL
    GROUP BY dept
    HAVING COUNT(*) >= 3
    ORDER BY avg_salary DESC
""").show()

In [ ]:
# SQL PIVOT equivalent (more limited than DataFrame pivot)
spark.sql("""
    SELECT dept,
        ROUND(AVG(CASE WHEN country='IN' THEN salary END), 0) AS IN_avg,
        ROUND(AVG(CASE WHEN country='UK' THEN salary END), 0) AS UK_avg,
        ROUND(AVG(CASE WHEN country='US' THEN salary END), 0) AS US_avg
    FROM employees
    GROUP BY dept
    ORDER BY dept
""").show()

# DataFrame pivot is cleaner for dynamic column values

## 10. HAVING Equivalent in DataFrame API

SQL has `HAVING` to filter groups **after** aggregation. In the DataFrame API, you just chain `.filter()` (or `.where()`) **after** `.agg()`.

In [ ]:
# SQL:  SELECT dept, COUNT(*) FROM employees GROUP BY dept HAVING COUNT(*) >= 3
# DF API equivalent:

df.groupBy("dept").agg(
    count("*").alias("headcount"),
    round(avg("salary"), 0).alias("avg_salary"),
).filter(col("headcount") >= 3) \
 .orderBy(col("avg_salary").desc()) \
 .show()

# The filter runs AFTER the groupBy — equivalent to HAVING

## 11. Performance: GroupBy Shuffle Cost

`groupBy` is a **wide transformation** — it causes a shuffle. The number of output partitions is controlled by `spark.sql.shuffle.partitions` (default: 200).

### Rules of thumb
| Data size | Recommended shuffle partitions |
|---|---|
| < 1 GB | 4–8 |
| 1–10 GB | 50–100 |
| 10–100 GB | 200–400 |
| > 100 GB | 400–2000 |

### Common groupBy mistakes
1. **High cardinality groupBy key** → many tiny groups → lots of partitions with little data
2. **Skewed key** → one group gets 90% of data → one slow reducer → stragglers
3. **collect_list on large groups** → driver OOM
4. **Default 200 partitions on small data** → too many tasks, overhead dominates

In [ ]:
# Check current setting
print("shuffle.partitions =", spark.conf.get("spark.sql.shuffle.partitions"))

# --- See the plan to understand the shuffle ---
df.groupBy("dept").agg(avg("salary")).explain()

# Look for:
# HashAggregate  ← partial agg on each partition (before shuffle)
# Exchange hashpartitioning  ← the shuffle
# HashAggregate  ← final agg after shuffle

In [ ]:
# --- Two-phase aggregation (what Spark does internally) ---
# Phase 1: Partial agg per partition (no shuffle)
# Phase 2: Shuffle + final agg
# This is called "partial aggregation" or "map-side combine" — same idea as combiner in MapReduce

# You can see both HashAggregates in explain():
# HashAggregate(keys=[dept], functions=[partial_avg(salary)])  ← phase 1
# Exchange hashpartitioning(dept, 4)                          ← shuffle
# HashAggregate(keys=[dept], functions=[avg(salary)])          ← phase 2

df.groupBy("dept").agg(avg("salary")).explain(mode="extended")

## 12. Real-world Pipeline: Department Analytics Report

Combine everything: groupBy, conditional agg, ranking, pivot.

In [ ]:
# --- Full analytics report ---

report = df.groupBy("dept").agg(
    count("*").alias("headcount"),
    count("salary").alias("reported_salary"),        # excludes nulls
    round(avg("salary"), 0).alias("avg_salary"),
    min("salary").alias("min_salary"),
    max("salary").alias("max_salary"),
    round(stddev("salary"), 0).alias("salary_stddev"),
    collect_set("country").alias("countries"),
    countDistinct("country").alias("n_countries"),
    count(when(col("hire_year") >= 2022, 1)).alias("new_hires_2022plus"),
    count(when(col("salary") > 90000, 1)).alias("senior_earners"),
).orderBy(col("avg_salary").desc())

report.show(truncate=False)

In [ ]:
# --- Cross-tab: salary band distribution per dept ---

banded = df.withColumn(
    "salary_band",
    when(col("salary") < 65000,  "<65k")
    .when(col("salary") < 80000, "65k-80k")
    .when(col("salary") < 95000, "80k-95k")
    .otherwise("95k+")
)

# Pivot to show band distribution per dept
banded.groupBy("dept") \
    .pivot("salary_band", ["<65k", "65k-80k", "80k-95k", "95k+"]) \
    .agg(count("*")) \
    .orderBy("dept") \
    .show()

## 13. Summary & Interview Cheat Sheet

### Core Pattern
```python
df.groupBy("col1", "col2")   # GroupedData (lazy)
  .agg(
      count("*").alias("n"),
      avg("salary").alias("avg_sal"),
  )                          # triggers shuffle
  .filter(col("n") > 1)     # HAVING equivalent
  .orderBy("avg_sal")
```

### Key Distinctions
| | |
|---|---|
| `count("*")` vs `count("col")` | `*` includes nulls; column skips nulls |
| `collect_list` vs `collect_set` | list has dups; set is unique (unordered) |
| `rollup` vs `cube` | rollup = left hierarchy; cube = all combinations |
| `pivot` without values | runs extra scan to find distinct values — always provide them |
| `first/last` | NOT deterministic without an explicit sort before grouping |

### Plan Reading
```
HashAggregate (partial)  ← Phase 1: map-side combine, no shuffle yet
Exchange                 ← shuffle — this is the cost
HashAggregate (final)    ← Phase 2: merge results per key
```

### Interview Questions
1. **What's the difference between `WHERE` and `HAVING` in Spark SQL?**  
   `WHERE` filters rows *before* grouping; `HAVING` filters groups *after* aggregation. In DataFrame API, `HAVING` = `.filter()` chained after `.agg()`.

2. **Why is `groupByKey` considered harmful in RDDs but `groupBy` in DataFrames is fine?**  
   DataFrame `groupBy` uses `HashAggregate` with partial (map-side) aggregation — data is pre-combined per partition before the shuffle, similar to a combiner. RDD `groupByKey` has no combiner — it shuffles all raw values. Catalyst handles the optimization automatically.

3. **When would you use `approx_count_distinct` over `countDistinct`?**  
   On very large datasets (100M+ rows), `countDistinct` requires keeping all values in memory for exact counting. `approx_count_distinct` uses HyperLogLog — ~2-5% error but O(1) memory. Use it for dashboards and monitoring where exact counts aren't required.

4. **What does `pivot` without a values list do internally?**  
   Spark first runs a separate job to collect all distinct values from the pivot column, then runs the actual pivot. Always provide explicit values to avoid this extra scan.

5. **What's the difference between `rollup` and `cube`?**  
   `rollup(A, B)` gives subtotals for (A,B), (A,null), (null,null) — hierarchical. `cube(A, B)` also adds (null,B) — all combinations. Use rollup for hierarchical data (year→month→day); use cube for multi-dimensional analysis.

## Exercises

Try these before looking at the solutions.

In [ ]:
# Exercise 1
# For each country, compute:
# - total headcount
# - average salary (round to 0 decimals)
# - number of distinct departments
# Filter to only countries with >= 3 employees
# Sort by avg_salary descending

# YOUR CODE HERE

In [ ]:
# Exercise 2
# Pivot: show headcount per dept per hire_year (years: 2019, 2020, 2021, 2022, 2023)
# Fill nulls with 0

# YOUR CODE HERE

In [ ]:
# Exercise 3
# For each dept, collect the list of employee names sorted alphabetically
# Hint: you need sort_array() from pyspark.sql.functions after collect_list

from pyspark.sql.functions import sort_array
# YOUR CODE HERE

In [ ]:
# Exercise 4
# Use rollup on (country, dept) to get a full salary report with subtotals
# Replace nulls in grouping columns with 'TOTAL'

# YOUR CODE HERE

### Solutions

In [ ]:
# Solution 1
df.groupBy("country").agg(
    count("*").alias("headcount"),
    round(avg("salary"), 0).alias("avg_salary"),
    countDistinct("dept").alias("n_depts"),
).filter(col("headcount") >= 3) \
 .orderBy(col("avg_salary").desc()) \
 .show()

In [ ]:
# Solution 2
from pyspark.sql.functions import coalesce

df.groupBy("dept") \
  .pivot("hire_year", [2019, 2020, 2021, 2022, 2023]) \
  .agg(count("*")) \
  .fillna(0) \
  .orderBy("dept") \
  .show()

In [ ]:
# Solution 3
df.groupBy("dept").agg(
    sort_array(collect_list("name")).alias("employees"),
).orderBy("dept").show(truncate=False)

In [ ]:
# Solution 4
df.rollup("country", "dept").agg(
    count("*").alias("headcount"),
    round(avg("salary"), 0).alias("avg_salary"),
).withColumn("country", coalesce(col("country"), lit("TOTAL"))) \
 .withColumn("dept",    coalesce(col("dept"),    lit("TOTAL"))) \
 .orderBy("country", "dept") \
 .show(40)

In [ ]:
spark.stop()